# Analisis Tren Harga Saham Bank Besar di Indonesia (2014–2024)

Midterm Project CAK4XBB3 Visualisasi Data · Kelompok [nomor]

| No | Nama | NIM |
|---|---|---|
| 1 | Aprilianza Muhammad Yusup | 103012300025 |
| 2 | Arfi | 103012300337 |
| 3 | Jeany | 103012300357 |
| 4 | Irgi | 103012300039 |
| 5 | Bill | [NIM] |

Sumber data: Dataset Saham Bank Indonesia 2014–2024 (Kaggle), https://www.kaggle.com/datasets/cmglonly/dataset-saham-bank-indonesia-2014-2024/data

## Load Data

Empat file CSV (BBCA, BBRI, BMRI, BBNI; sumber Yahoo Finance) digabung menjadi `df_all`, lalu dibuat `adj_wide` (Adj Close) dan `vol_wide` (Volume) dengan tanggal sebagai index.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

TICKERS = ['BBCA', 'BBRI', 'BMRI', 'BBNI']
NAMA_BANK = {
    'BBCA': 'Bank Central Asia',
    'BBRI': 'Bank Rakyat Indonesia',
    'BMRI': 'Bank Mandiri',
    'BBNI': 'Bank Negara Indonesia',
}
WARNA = {'BBCA': '#2a78d6', 'BBRI': '#eb6834', 'BMRI': '#1baf7a', 'BBNI': '#eda100'}

# lokasi folder dataset (root repo / folder kerja), kalau tidak ada ambil dari GitHub
if Path('dataset').is_dir():
    ROOT = Path('.')
elif Path('../dataset').is_dir():
    ROOT = Path('..')
else:
    ROOT = None
DATA_DIR = f'{ROOT}/dataset' if ROOT else 'https://raw.githubusercontent.com/aprilianza/Midterm_DataVisualization/main/dataset'
FIG_DIR = (ROOT or Path('.')) / 'figures'
FIG_DIR.mkdir(exist_ok=True)

list_df = []
for t in TICKERS:
    df = pd.read_csv(f'{DATA_DIR}/{t}.JK_10_years.csv')
    df['Date'] = pd.to_datetime(df['Date'], utc=True).dt.tz_localize(None)
    df['Ticker'] = t
    list_df.append(df.sort_values('Date'))

df_all = pd.concat(list_df, ignore_index=True)
df_all = df_all[['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']]
df_all = df_all.drop_duplicates(['Ticker', 'Date'])

# baris volume 0 dengan harga datar = hari libur bursa, dibuang untuk semua bank
libur = ((df_all['Volume'] == 0) & (df_all['Open'] == df_all['Close'])
         & (df_all['High'] == df_all['Close']) & (df_all['Low'] == df_all['Close']))
HARI_DIBUANG = pd.DatetimeIndex(df_all.loc[libur, 'Date'].unique()).sort_values()
df_all = df_all[~df_all['Date'].isin(HARI_DIBUANG)].reset_index(drop=True)

adj_wide = df_all.pivot(index='Date', columns='Ticker', values='Adj Close')[TICKERS]
vol_wide = df_all.pivot(index='Date', columns='Ticker', values='Volume')[TICKERS]

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'savefig.dpi': 200,
    'savefig.bbox': 'tight',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.titlelocation': 'left',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.color': '#e6e5e0',
    'legend.frameon': False,
})


def simpan_fig(fig, nama):
    fig.savefig(FIG_DIR / f'{nama}.png')


def format_tanggal(ax):
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))


print('Jumlah baris df_all :', len(df_all))
print('Rentang tanggal     :', df_all['Date'].min().date(), 's.d.', df_all['Date'].max().date())
print('Hari libur dibuang  :', len(HARI_DIBUANG))

Jumlah baris df_all : 9616
Rentang tanggal     : 2014-12-08 s.d. 2024-12-03
Hari libur dibuang  : 69


## Profil Dataset

### Jumlah Observasi dan Rentang Waktu

In [2]:
ringkas = df_all.groupby('Ticker', sort=False).agg(
    Observasi=('Date', 'size'),
    Mulai=('Date', 'min'),
    Selesai=('Date', 'max'),
)
ringkas.insert(0, 'Nama', ringkas.index.map(NAMA_BANK))
ringkas

,Nama,Observasi,Mulai,Selesai
Ticker,,,,
BBCA,Bank Central Asia,2404,2014-12-08,2024-12-03
BBRI,Bank Rakyat Indonesia,2404,2014-12-08,2024-12-03
BMRI,Bank Mandiri,2404,2014-12-08,2024-12-03
BBNI,Bank Negara Indonesia,2404,2014-12-08,2024-12-03


### Missing Value dan Duplikat

Pada data mentah terdapat baris dengan Volume 0 dan harga Open, High, Low, Close yang sama. Tanggal baris tersebut bertepatan dengan hari libur bursa seperti 17 Agustus, Natal, Tahun Baru, dan cuti bersama Lebaran, sehingga bukan hari perdagangan. Baris ini sudah dibuang saat load data (disimpan di `HARI_DIBUANG`).

In [3]:
print(len(HARI_DIBUANG), 'tanggal dibuang, contoh:')
print(HARI_DIBUANG[:10].strftime('%d %b %Y (%a)').tolist())

69 tanggal dibuang, contoh:
['09 Dec 2015 (Wed)', '11 Mar 2016 (Fri)', '14 Mar 2016 (Mon)', '11 Apr 2016 (Mon)', '13 Apr 2016 (Wed)', '14 Apr 2016 (Thu)', '15 Apr 2016 (Fri)', '18 Apr 2016 (Mon)', '19 Apr 2016 (Tue)', '02 Jan 2017 (Mon)']


In [4]:
print('Missing value per kolom:')
print(df_all.isna().sum(), end='\n\n')
print('Duplikat (Ticker, Date):', df_all.duplicated(['Ticker', 'Date']).sum())
print('Tanggal yang tidak lengkap di 4 bank:', adj_wide.isna().any(axis=1).sum())
print('Sisa hari dengan Volume = 0:')
print((vol_wide == 0).sum())

Missing value per kolom:
Date         0
Ticker       0
Open         0
High         0
Low          0
Close        0
Adj Close    0
Volume       0
dtype: int64

Duplikat (Ticker, Date): 0
Tanggal yang tidak lengkap di 4 bank: 0
Sisa hari dengan Volume = 0:
Ticker
BBCA    0
BBRI    0
BMRI    0
BBNI    0
dtype: int64


### Granularitas Waktu

Data bersifat harian, tetapi hanya mencakup hari bursa (tanpa Sabtu, Minggu, dan hari libur). Tahun 2014 dan 2024 tidak penuh satu tahun.

In [5]:
hari_bursa = adj_wide.groupby(adj_wide.index.year).size().rename('Hari bursa')
hari_bursa.to_frame().T

Date,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
Hari bursa,15,244,238,237,233,245,240,247,246,239,220


In [6]:
tgl = adj_wide.index.to_series()
selisih = tgl.diff().dt.days
print('Jarak antar observasi (hari):')
print(selisih.value_counts().sort_index().head(10), end='\n\n')
print('5 jeda terpanjang:')
pd.DataFrame({'Dari': tgl.shift(), 'Sampai': tgl, 'Hari': selisih}).nlargest(5, 'Hari').reset_index(drop=True)

Jarak antar observasi (hari):
Date
1.0     1833
2.0       51
3.0      446
4.0       38
5.0       21
6.0        5
7.0        1
8.0        2
10.0       2
11.0       3
Name: count, dtype: int64

5 jeda terpanjang:


,Dari,Sampai,Hari
0,2018-06-08,2018-06-20,12.0
1,2017-06-22,2017-07-03,11.0
2,2022-04-28,2022-05-09,11.0
3,2024-04-05,2024-04-16,11.0
4,2016-07-01,2016-07-11,10.0


### Statistik Deskriptif

In [7]:
adj_wide.describe().round(0)

Ticker,BBCA,BBRI,BMRI,BBNI
count,2404.0,2404.0,2404.0,2404.0
mean,5451.0,2994.0,3179.0,3245.0
std,2467.0,1261.0,1491.0,1037.0
min,1925.0,1074.0,1278.0,1409.0
25%,3138.0,1956.0,2057.0,2319.0
50%,5360.0,2892.0,2746.0,3132.0
75%,7356.0,3918.0,3621.0,3983.0
max,10896.0,6300.0,7450.0,5950.0


In [8]:
# volume dalam juta lembar
(vol_wide.describe().drop('count') / 1e6).round(1)

Ticker,BBCA,BBRI,BMRI,BBNI
mean,83.5,157.4,96.0,60.0
std,54.4,98.6,55.9,43.4
min,10.7,27.7,6.0,6.8
25%,54.3,95.3,59.7,34.0
50%,72.1,130.7,82.9,48.1
75%,98.4,186.7,116.2,71.5
max,1062.9,959.9,770.3,444.1


### Close vs Adj Close

Pada data Yahoo Finance, kolom `Close` sudah disesuaikan dengan stock split, sedangkan `Adj Close` disesuaikan dengan stock split dan dividen. Karena penyesuaian split, kolom `Close` memiliki harga pecahan (misalnya 2.987,5) pada periode sebelum split, padahal harga saham di BEI selalu bilangan bulat. Untuk membandingkan pertumbuhan harga antar bank, analisis selanjutnya menggunakan `Adj Close`.

In [9]:
rasio = (df_all['Adj Close'] / df_all['Close']).groupby(df_all['Ticker'], sort=False)
pd.DataFrame({'Rasio awal': rasio.first(), 'Rasio akhir': rasio.last()}).round(3)

,Rasio awal,Rasio akhir
Ticker,,
BBCA,0.846,1.0
BBRI,0.696,1.0
BMRI,0.668,1.0
BBNI,0.762,1.0


In [10]:
pecahan = df_all[df_all['Close'] % 1 != 0]
pecahan.groupby('Ticker', sort=False)['Date'].agg(['size', 'max']).rename(
    columns={'size': 'Hari Close pecahan', 'max': 'Terakhir muncul'})

,Hari Close pecahan,Terakhir muncul
Ticker,,
BBRI,1618,2021-09-07
BMRI,1025,2023-04-03
BBNI,844,2023-10-05


## Line Plot

## Area Plot

## Multiple Time Series

## Seasonal Plot / Calendar Heatmap

## Moving Average

## Fitting Model

## Deklarasi Penggunaan AI

"Saya, [Nama Lengkap] / [NIM], menyatakan bahwa saya [menggunakan / tidak menggunakan] alat AI generatif dalam pengerjaan tugas ini. Seluruh interpretasi, insight, dan kesimpulan yang disampaikan merupakan hasil pemahaman saya sendiri."

| No | Nama Tools AI | Bagian yang Dibantu | Bentuk Bantuan / Tujuan Prompt | Tingkat Ketergantungan |
|---|---|---|---|---|
| 1 | | | | |

Screenshot prompt dikumpulkan terpisah dalam berkas `Screenshot_AI_[NIM]`.